# ODAC25 + UMA authenticated smoke test
Attach the private project-bundle Dataset, enable Internet and one GPU, and attach a Kaggle secret named `HF_TOKEN`. Run cells in order.

In [ ]:
%pip install -q "huggingface-hub>=0.28" "ase>=3.26" "fairchem-core>=2,<3" "fairchem-data-odac>=0.1,<1"

In [ ]:
import os, subprocess, sys, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
assert os.environ['HF_TOKEN'], 'Attach the HF_TOKEN secret to this notebook'

hits = list(Path('/kaggle/input').rglob('scripts/odac25_hf.py'))
if hits:
    REPO = hits[0].parents[1]
else:
    archives = list(Path('/kaggle/input').rglob('MOF_DAC_Kaggle_bundle.zip'))
    if len(archives) != 1:
        raise FileNotFoundError('Attach exactly one MOF DAC bundle Dataset')
    REPO = Path('/kaggle/working/MOF_for_DAC_Opt')
    REPO.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as bundle:
        bundle.extractall(REPO)
assert (REPO / 'scripts/odac25_hf.py').is_file(), REPO
os.chdir(REPO)
print('repository:', REPO)

In [ ]:
OUT = Path('/kaggle/working/odac25')
OUT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, 'scripts/odac25_hf.py', 'inventory', '--output', str(OUT/'inventory.json')], check=True)

In [ ]:
import json
inventory = json.loads((OUT/'inventory.json').read_text())
print('revision:', inventory['revision'])
print('repository files:', len(inventory['files']))
print((OUT/'DATASET.md').read_text()[:12000])


## Bounded validation-shard extraction
The gated repository contains metadata and official archive links. Stream one 0.79 GiB GCMC validation member without retaining the 12 GB archive. This shard validates schema and UMA execution only.

In [ ]:
import shutil, tarfile, urllib.request

VAL_URL = 'https://dl.fbaipublicfiles.com/dac/odac25/20250918/odac25_filtered_val.tar.gz'
WANTED = {
    'val/gcmc/part_00000.aselmdb-lock',
    'val/gcmc/part_00000.aselmdb',
}
data_root = OUT/'data'
extracted = []
with urllib.request.urlopen(VAL_URL, timeout=300) as response:
    with tarfile.open(fileobj=response, mode='r|gz') as archive:
        for member in archive:
            if member.name not in WANTED:
                continue
            if member.size > 2*1024**3:
                raise RuntimeError(f'Unexpected member size: {member.size}')
            destination = data_root/member.name
            destination.parent.mkdir(parents=True, exist_ok=True)
            source = archive.extractfile(member)
            if source is None:
                raise RuntimeError(f'Cannot extract {member.name}')
            with source, destination.open('wb') as sink:
                shutil.copyfileobj(source, sink, length=8*1024*1024)
            extracted.append(member.name)
            print('extracted', member.name, member.size)
            if set(extracted) == WANTED:
                break
assert set(extracted) == WANTED, extracted
DATASET_SOURCE = data_root/'val/gcmc'
print('dataset source:', DATASET_SOURCE)


In [ ]:
subprocess.run([sys.executable, 'scripts/odac25_hf.py', 'inspect', str(DATASET_SOURCE), '--output', str(OUT/'schema.json'), '--samples', '16'], check=True)
print((OUT/'schema.json').read_text())

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'scripts/odac25_hf.py', 'uma-smoke', str(DATASET_SOURCE), '--output', str(OUT/'uma_smoke.json'), '--samples', '16', '--device', 'cuda'], check=True)
print((OUT/'uma_smoke.json').read_text())

In [ ]:
deliverables = ['inventory.json', 'DATASET.md', 'schema.json', 'uma_smoke.json']
for name in deliverables:
    assert (OUT/name).is_file(), name
archive = Path('/kaggle/working/ODAC25_UMA_smoke_artifacts.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in deliverables:
        zf.write(OUT/name, arcname=name)
print('Download from Notebook Output:', archive)